In [1]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import torch
import torchaudio
import transformers
print(torch.__version__)
print(torchaudio.__version__)
print(transformers.__version__)

1.13.1+cu117
0.13.1+cu117
4.26.1


In [6]:
from os                import walk
from pydub             import AudioSegment
from pydub.utils       import get_array_type
from pydub.utils       import mediainfo
from pydub.silence     import split_on_silence
from datasets          import load_dataset
from transformers      import Wav2Vec2ForCTC, Wav2Vec2Processor
from transformers      import WhisperProcessor, WhisperForConditionalGeneration
from torchaudio.utils  import download_asset
from scipy             import signal
from scipy.io          import wavfile
from matplotlib.pyplot import figure
from tqdm              import tqdm
from os                import listdir
from os.path           import isfile, join
from datetime          import datetime
from pyctcdecode       import build_ctcdecoder
from pprint            import pprint

import matplotlib.pyplot as plt
import pandas            as pd
import numpy             as np
import soundfile         as sf

import git
import whisper
import os
import jiwer
import IPython
import array
import librosa
import torch
import torchaudio

kenlm python bindings are not installed. Most likely you want to install it using: pip install https://github.com/kpu/kenlm/archive/master.zip
kenlm python bindings are not installed. Most likely you want to install it using: pip install https://github.com/kpu/kenlm/archive/master.zip


<h1 style="background-color:LightGreen;"> <center> <a id='start_cell'></a> Table Of Contents </center></h1>

[Prepare Data](#prepare_data_cell) </br>
[Run Validation Tests](#run_validation_tests) </br>
[GIT](#add_to_git) </br>
[Play Audio](#play_audio_cell) </br>
[whisper](#whisper_cell) </br>
[Wav2Vec2](#wav2vec2_cell) </br>
[Compare Wav2Vec2 VS Whisper](#compare_cell) </br>
[Q's to shiri](#shiri_cell) </br>
[Beam Search Tests](#bs_cell) </br>

<h1 style="background-color:LightGreen;"> <center> Prepare Data </center></h1>

https://towardsdatascience.com/transcribe-audio-files-with-openais-whisper-e973ae348aa7

In [11]:
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cuda', index=0)

<h1 style="background-color:LightGreen;"> <center> <a id='prepare_data_cell'></a> Prepare Data </center></h1>

In [12]:
WAV_FULL_FILE_FOLDER   = "/home/amitli/Datasets/Voice/Hebrew/Police"
WAV_SPLIT_SR_8_FOLDER  = "/home/amitli/Datasets/MyVoiceTests"
WAV_SPLIT_SR_16_FOLDER = "/home/amitli/Datasets/MyVoiceTests_16SR"
df_gt                  = pd.read_excel("/home/amitli/Datasets/Voice/Hebrew/new_corpus.xlsx")

In [13]:
def get_part_of_wav(file_path, start_time_sec, end_time_sec, new_file_path):    
    t1       = start_time_sec * 1000 
    t2       = end_time_sec * 1000
    newAudio = AudioSegment.from_wav(file_path)    
    newAudio = newAudio[t1:t2]        
    newAudio.export(new_file_path, format="wav")

In [14]:
def split_file(arr_gt_times, wav_file_path, file_name, split_folder):
    '''
        arr_gt_times   - times to split file: 'wav_file_path/file_name'
        wav_file_path  - path only (without name of file)
        file_name      - file name to split
        split_folder   - output folder
    '''
    
    for i in range(len(arr_gt_times)):
        
            times      = arr_gt_times[i]
            start_time = times[0:times.find(" ")]
            end_time   = times[times.find(" ")+2:].lstrip()

            time_obj   =  datetime.strptime(start_time, '%H:%M:%S.%f').time()
            start_time = (time_obj.hour * 3600000) + (time_obj.minute * 60000) + (time_obj.second * 1000) + (time_obj.microsecond / 1000)

            time_obj   =  datetime.strptime(end_time, '%H:%M:%S.%f').time()
            end_time   = (time_obj.hour * 3600000) + (time_obj.minute * 60000) + (time_obj.second * 1000) + (time_obj.microsecond / 1000)
                        
            try:
                path       = os.path.join(split_folder, file_name)             
                os.mkdir(path, 0o777)
            except:
                None
            
            file_num   = "%04d" % (i,)
            get_part_of_wav(file_path      = f"{wav_file_path}/{file_name}.wav", 
                            start_time_sec = start_time / 1000, 
                            end_time_sec   = end_time / 1000, 
                            new_file_path  = f"{split_folder}/{file_name}/{file_name}_{file_num}.wav")
        

In [15]:
def split_all_wav_files(df, source_folder, dest_folder):
    files_to_split = [f for f in listdir(source_folder) if isfile(join(source_folder, f))]
    for i, file_to_split in tqdm(enumerate(files_to_split)):
        
        file_name    = file_to_split[:-4]        
        arr_gt_times = np.array(df[df['file'] == f"{file_name}.mp3"]['segment'])        
        
        split_file(arr_gt_times, source_folder, file_name, dest_folder)
                

In [16]:
def convert_to_16sr_file(source_path, dest_path):
    
    speech, sr = librosa.load(source_path, sr=16000)
    sf.write(dest_path, speech, sr)

def convert_to_16_sr_folder(source_folder, dest_folder):    
    
    folder_list = [f for f in listdir(source_folgitder) if not isfile(join(source_folder, f))]        
    
    for folder_name in folder_list:
        splitted_wav_folder = f"{source_folder}/{folder_name}"
        splitted_files      = [entry for entry in os.listdir(splitted_wav_folder) if os.path.isfile(os.path.join(splitted_wav_folder, entry))]
        for i in range(len(splitted_files)): 
            part          = "%04d" % (i,)
            full_src      = f"{source_folder}/{folder_name}/{folder_name}_{part}.wav"
            full_dst      = f"{dest_folder}/{folder_name}/{folder_name}_{part}.wav"
            
            try:                
                os.mkdir(f"{dest_folder}/{folder_name}", 0o777)
            except:
                None
            
            convert_to_16sr_file(full_src, full_dst)        

In [17]:
SPLIT_FILES      = False
CONVERT_TO_SR_16 = False

if SPLIT_FILES:
    split_all_wav_files(df_gt, WAV_FULL_FILE_FOLDER, WAV_SPLIT_SR_8_FOLDER)    
    
if CONVERT_TO_SR_16:
    convert_to_16_sr_folder(WAV_SPLIT_SR_8_FOLDER, WAV_SPLIT_SR_16_FOLDER)    

In [18]:
def get_sample_rate(file):
    info      = mediainfo(file)
    return int(info['sample_rate'])

In [19]:
def get_wav_duration(wav_file):
    res = librosa.get_duration(path=wav_file)
    return res

<h1 style="background-color:LightGreen;"> <center> <a id='run_validation_tests'></a>  Run Validation Tests </center></h1>

[Table Of Content](#start_cell)

In [20]:
def get_total_speaking_length(speech_timestamps):
    total = 0
    for val in speech_timestamps:
        tmp = val['end'] - val['start']
        total = total + tmp
    return total

In [ ]:
TEST_VAD = True
if TEST_VAD is True:
    
    
    vad_model, utils = torch.hub.load(repo_or_dir  = 'snakers4/silero-vad',
                                  model        = 'silero_vad',
                                  force_reload = True,
                                  onnx         = False)

    (get_speech_timestamps, save_audio, read_audio,VADIterator, collect_chunks) = utils
    
    
    source_folder = WAV_SPLIT_SR_16_FOLDER
    folder_list   = [f for f in listdir(source_folder) if not isfile(join(source_folder, f))]        
    df_vad        = pd.DataFrame()

    file_names        = []
    file_duration     = []
    file_num_of_words = []
    vad_speaking_ms   = []    
    vad_speaking_prec = []
    
    for folder_name in tqdm(folder_list):
        
        splitted_wav_folder = f"{source_folder}/{folder_name}"
        splitted_files      = [entry for entry in os.listdir(splitted_wav_folder) if os.path.isfile(os.path.join(splitted_wav_folder, entry))]
        gt_list             = df_gt[df_gt['file'] == f"{folder_name}.mp3"]['text'].values        
        
                        
        for i in range(len(splitted_files)): 
            part           = "%04d" % (i,)
            full_src       = f"{source_folder}/{folder_name}/{folder_name}_{part}.wav"                        

            sample_rate       = 8000
            wav_audio         = read_audio(full_src, sampling_rate=sample_rate)            
            speech_timestamps = get_speech_timestamps(wav_audio, vad_model, sampling_rate=sample_rate, return_seconds=True)
            
            wav_duration   = get_wav_duration(full_src)
            text_length    = len(gt_list[i].split())
                        

            file_duration.append(wav_duration)
            file_num_of_words.append(text_length)
            file_names.append(f"{folder_name}_{part}")
                                
            total_speaking = get_total_speaking_length(speech_timestamps)
            vad_speaking_ms.append(total_speaking)
            vad_speaking_prec.append(total_speaking/wav_duration)
        
    df_vad = pd.DataFrame(data = {"File":                 file_names,
                                  "Duration (Sec)":       file_duration,
                                  "Text Length":          file_num_of_words,
                                  "Speaking Length (ms)": vad_speaking_ms,
                                  "Spealing Percentage":  vad_speaking_prec})
    df_vad.to_csv("/home/amitli/Repo/Voice/Results/vad.csv")
    add_to_git()

Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /home/amitli/.cache/torch/hub/master.zip
  0%|                                                                                                                                                                              | 0/157 [00:00<?, ?it/s]/home/amitli/venv310/lib/python3.8/site-packages/torch/nn/modules/module.py:1194: UserWarning: operator() profile_node %106 : int[] = prim::profile_ivalue(%104)
 does not have profile information (Triggered internally at ../torch/csrc/jit/codegen/cuda/graph_fuser.cpp:105.)
  return forward_call(*input, **kwargs)
 92%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍             | 144/157 [21:54<03:04, 14.23s/it]

In [ ]:
df_vad = pd.read_csv("/home/amitli/Repo/Voice/Results/vad.csv")

In [ ]:
df_vad[df_vad['Vad Value'] < 0.3].head()

In [ ]:
IPython.display.Audio("/home/amitli/Datasets/MyVoiceTests_16SR/0XLKQME2ZJc/0XLKQME2ZJc_0088.wav")

In [ ]:
df_vad[df_vad['Duration (Sec)'] *4 < df_vad['Text Length']].head()

In [ ]:
get_wav_duration("/home/amitli/Datasets/MyVoiceTests_16SR/-3dd58g7rQQ/-3dd58g7rQQ_0002.wav")

In [ ]:
# model, utils = torch.hub.load('snakers4/silero-vad', 'silero_vad')

# (get_speech_timestamps, _, read_audio, *_) = utils

# sampling_rate = 16000 
# wav = read_audio("/home/amitli/Datasets/MyVoiceTests_16SR/-3dd58g7rQQ/-3dd58g7rQQ_0002.wav", sampling_rate=sampling_rate)


# speech_timestamps = get_speech_timestamps(wav, model, sampling_rate=sampling_rate)
# pprint(speech_timestamps)

<h1 style="background-color:LightGreen;"> <center> <a id='add_to_git'></a>  Add To GIT </center></h1>

In [7]:
def add_to_git():
    repo = git.Repo("/home/amitli/Repo/Voice")
    repo.index.add(["Main_Notebook.ipynb", "Results/*"])
    repo.index.commit("Add new results")
    origin = repo.remote(name='origin')
    origin.push()

In [10]:
add_to_git()

<h1 style="background-color:LightGreen;"> <center> <a id='play_audio_cell'></a>  Play Audio </center></h1>

[Table Of Content](#start_cell)

In [ ]:
#IPython.display.Audio(PART)

<h1 style="background-color:LightGreen;"> <center> <a id='whisper_cell'></a> Whisper </center></h1>

[Table Of Content](#start_cell)

In [ ]:
def Get_HF_Whisper_Text(whisper_model, whisper_processor, file_name, device):   
    forced_decoder_ids = whisper_processor.get_decoder_prompt_ids(language="hebrew", task="transcribe")

    speech, sr     = librosa.load(file_name, sr=16000)
    inputs         = whisper_processor(speech, 
                                       sampling_rate  = sr, 
                                       return_tensors = "pt")
        
    input_features = inputs.input_features.to(device)     
    
    #with torch.no_grad():
    predicted_ids  = whisper_model.generate(input_features, forced_decoder_ids=forced_decoder_ids)            
    transcription  = whisper_processor.batch_decode(predicted_ids, skip_special_tokens=True)
    
    return transcription[0]    
# print(f"WER Whisper: {wer_whisper}")


In [ ]:
def Get_Whisper_Text(whisper_model, whisper_processor, file_name, device):
    
    # load audio and pad/trim it to fit 30 seconds
    audio = whisper.load_audio(file_name)
    audio = whisper.pad_or_trim(audio)
    mel   = whisper.log_mel_spectrogram(audio).to(whisper_model.device)

    # decode the audio
    options = whisper.DecodingOptions(language = 'he', beam_size=8, patience=2, fp16 = False)    
    result  = whisper.decode(whisper_model, mel, options)
    
    return result.text

In [ ]:
WHISPER_OPEN_AI = True

if WHISPER_OPEN_AI is True:
    whisper_model     = whisper.load_model("large", device="cpu")
    whisper_processor = None
else:
    whisper_model_name = "openai/whisper-large"
    whisper_model      = WhisperForConditionalGeneration.from_pretrained(whisper_model_name)
    whisper_model.to("cpu")
    whisper_processor  = WhisperProcessor.from_pretrained(whisper_model_name)


<h1 style="background-color:LightGreen;"> <center> <a id='wav2vec2_cell'></a>  Wav2Vec2 </center></h1>

[Table Of Content](#start_cell)

In [ ]:
def decode_hebrew(transcription):
    heb = ""
    for ch in transcription:
        if (ch == '[') or (ch == 'P') or (ch == 'A') or (ch == 'D') or (ch == ']'):
            continue
        heb = heb + ch
    return heb

In [ ]:
def Get_Wav2Vec2_Text(wav2vec_model, wav2vec2_processor, file_name, device):
    speech, sr       = librosa.load(file_name, sr=16000)

    inputs           = wav2vec2_processor(speech, sampling_rate=16000, return_tensors="pt")
    input_values     = inputs.input_values.to(device)     
    with torch.no_grad():
        logits           = wav2vec_model(input_values).logits
        
    if False:
        # Beam search - check why it works worse than naive method
        vocabulary       = list(wav2vec2_processor.tokenizer.get_vocab().keys())
        decoder          = build_ctcdecoder(vocabulary)
        hebrew_text      = decoder.decode(logits[0].numpy(), beam_width=8)
        
    if True:
        predicted_ids    = torch.argmax(logits, dim=-1)
        transcription    = wav2vec2_processor.decode(predicted_ids[0])
        hebrew_text      = decode_hebrew(transcription)
    
    return hebrew_text

In [ ]:
model_name         = "imvladikon/wav2vec2-xls-r-300m-hebrew"
wav2vec2_model     = Wav2Vec2ForCTC.from_pretrained(model_name)    
wav2vec2_model.to("cpu")
wav2vec2_processor = Wav2Vec2Processor.from_pretrained(model_name)

<h1 style="background-color:LightGreen;"> <center> <a id='compare_cell'></a>  Compare </center></h1>

[Table Of Content](#start_cell)

In [ ]:
def count_num_of_words(arr_asr_res):
    res = []
    for item in arr_asr_res:
        res.append(len(item.split()))
    return res

In [ ]:
def comapre_ASR_models(whisper_model, whisper_processor, wav2vec2_model, wav2vec2_processor, gt_list, post_name, test_path):
    
    arr_whisper     = []
    arr_wav2vec2    = []
    arr_gt          = []      
    arr_wav_len_sec = []
    num_of_files    = len([entry for entry in os.listdir(test_path) if os.path.isfile(os.path.join(test_path, entry))])    
    
    for i in range(num_of_files):
        
        part          = "%04d" % (i,)
        test_file     = f"{test_path}{post_name}_{part}.wav"          
        wav_duration  = get_wav_duration(test_file)        
                
        whisper_text     = Get_Whisper_Text(whisper_model, whisper_processor, test_file, "cpu")
        #whisper_text     = Get_HF_Whisper_Text(whisper_model, whisper_processor, test_file, "cpu") 
        wav2vec2_text     = Get_Wav2Vec2_Text(wav2vec2_model, wav2vec2_processor, test_file, "cpu")        
                
        arr_whisper.append(whisper_text)
        arr_wav2vec2.append(wav2vec2_text)
        arr_gt.append(gt_list[i])
        arr_wav_len_sec.append(wav_duration)
                        
    return arr_whisper, arr_wav2vec2, arr_gt, arr_wav_len_sec

In [ ]:
def calc_cer_per_sentence(gt_list, model_list):
    res = []
    for i in range(len(gt_list)):
        gt_text    = gt_list[i]
        model_text = model_list[i]
        cer        = jiwer.cer(gt_text, model_text)        
        res.append(cer)
    return res

In [ ]:
def calc_wer_per_sentence(gt_list, model_list):
    res = []
    for i in range(len(gt_list)):
        gt_text    = gt_list[i]
        model_text = model_list[i]
        wer        = jiwer.wer(gt_text, model_text)        
        res.append(wer)
    return res

In [ ]:
def create_file_name_title(file_name, num_of_items):
    res = []
    for i in range(num_of_items):
        part = "%04d" % (i,)
        res.append(f"{file_name}_{part}")
    return res

In [ ]:
def compare_models(wav_folder, compare_list, df_desc, df_result_path, whisper_model, whisper_processor, wav2vec2_model, wav2vec2_processor):
            
        
    for i, file_to_compare in tqdm(enumerate(compare_list)):
        
        gt_list = df_desc[df_desc['file'] == f"{file_to_compare}.mp3"]['text'].values        
        
        arr_whisper, arr_wav2vec2, arr_gt, arr_durations = comapre_ASR_models(whisper_model, 
                                                               whisper_processor, 
                                                               wav2vec2_model,
                                                               wav2vec2_processor,                                             
                                                               gt_list,  
                                                               file_to_compare, 
                                                               f"{wav_folder}/{file_to_compare}/")
        
        
        
        whisper_word_cnt  = count_num_of_words(arr_whisper)
        w2v_word_cnt      = count_num_of_words(arr_wav2vec2)
        gt_word_cnt       = count_num_of_words(arr_gt)
                
                
        wer_wav2vec2 = calc_wer_per_sentence(gt_list, arr_wav2vec2)
        wer_whisper  = calc_wer_per_sentence(gt_list, arr_whisper)
        
        cer_wav2vec2 = calc_cer_per_sentence(gt_list, arr_wav2vec2)
        cer_whisper  = calc_cer_per_sentence(gt_list, arr_whisper)
        
        names        = create_file_name_title(file_to_compare, len(gt_list))
                
        df_tmp       = pd.DataFrame(data = {"File":           names,
                                            "Duration (Sec)":     arr_durations,                                            
                                            "GT":                 gt_list,
                                            "Wav2Vec2_Text":      arr_wav2vec2,
                                            "Whisper_Text":       arr_whisper,
                                            "Wav2Vec2_WER":       wer_wav2vec2,
                                            "Whisper_WER":        wer_whisper,
                                            "Wav2Vec2_CER":       cer_wav2vec2,
                                            "Whisper_CER":        cer_whisper,
                                            "GT Words Len":       gt_word_cnt,
                                            "Wav2Vec2 Words Len": w2v_word_cnt,
                                            "Whisper Words Len":  whisper_word_cnt,})
        df_tmp.to_csv(f"{DF_RESULT_PATH}/{file_to_compare}.csv")
        add_to_git()
        

In [ ]:
DF_RESULT_PATH = "/home/amitli/Repo/Voice/Results"

In [ ]:
FOLDER_LIST = ['2RpZkeaTIm4', '1KpCbxgsGUU', '0XLKQME2ZJc', '0WTBFu4kJyQ', '1Ls2KdQYYu4', '1SlSJ5YeyfY', '2KoqxFUJd2c', '4PprAxJ2KPs', '3yDUlw5C5wk', '0HXHLe9GNpU', '0RN1xVXOemU', '2yZfyjjpp5Y', '-3dd58g7rQQ', '3gNAqdgwJPo', '4SkK1VrkJKc', '4_GQkF3MiIU', '2-gLlE-j3_U', '4A-SzBqUZ0E', '2eaXPCSUZbQ', '3WR5XZMvz8I']
len(FOLDER_LIST)

In [ ]:
RUN_COMPARE = True
if RUN_COMPARE is True:
    compare_models(WAV_SPLIT_SR_16_FOLDER, FOLDER_LIST, df_gt, DF_RESULT_PATH, whisper_model, whisper_processor, wav2vec2_model, wav2vec2_processor)            
    #compare_models(WAV_SPLIT_SR_16_FOLDER, ["0qGUNd5488I"], df_gt, DF_RESULT_PATH, whisper_model, whisper_processor, wav2vec2_model, wav2vec2_processor)    

[Whisper](#whisper_cell) </br>
[Wav2Vec2](#wav2vec2_cell) </br>

In [ ]:
source_folder = WAV_SPLIT_SR_16_FOLDER
folder_list   = [f for f in listdir(source_folder) if not isfile(join(source_folder, f))]        
print(folder_list)

In [ ]:
csv_file_naive       = "/home/amitli/Datasets/df_res.csv"
#csv_file_beam_search = "/home/amitli/Datasets/df_res_openai_bs.csv"
csv_file_beam_search = "/home/amitli/Datasets/df_res_openai_bs_large.csv"
csv_file_wav2vec2_bs = "/home/amitli/Datasets/df_res_wav2vec2_bs.csv"
df_res_wav_bs        = pd.read_csv(csv_file_wav2vec2_bs)
df_res_naive         = pd.read_csv(csv_file_naive)
df_res_beam_search   = pd.read_csv(csv_file_beam_search)
df_res_wav_bs.head(10)

In [ ]:
naive_Whisper_text = df_res_naive['Whisper_Text'].values
bs_Whisper_text    = df_res_beam_search['Whisper_Text'].values
gt_text            = df_res_beam_search['GT'].values
wav2vec2_text      = df_res_wav_bs['Wav2Vec2_Text'].values


In [ ]:
wav2vec2_wer      = jiwer.wer(list(gt_text), list(wav2vec2_text))
naive_Whisper_wer = jiwer.wer(list(gt_text), list(naive_Whisper_text))
bs_Whisper_wer    = jiwer.wer(list(gt_text), list(bs_Whisper_text))

print(f"Wav2Vec2 WER:      {wav2vec2_wer}")
print(f"naive Whisper WER: {naive_Whisper_wer}")
print(f"bs Whisper: WER:   {bs_Whisper_wer}")

In [ ]:
# Wav2Vec2 WER:      0.7964601769911505
# naive Whisper WER: 1.0206489675516224 (small HF)
# bs Whisper: WER:   1.0206489675516224  (openAI small beam search)

In [ ]:
df = pd.read_csv("/home/amitli/Datasets/df_res_openai_bs_large.csv")
df.head(5)

<h1 style="background-color:#B61FEB;"> <center> <a id='shiri_cell'></a>  Check With SHIRI: </center></h1>

[Table Of Content](#start_cell)

<h1 style="background-color:LightGreen;"> <center> <a id='bs_cell'></a>  Beam Search </center></h1>

[Table Of Content](#start_cell)

In [ ]:
#IPython.display.Audio("/home/amitli/Datasets/MyVoiceTests_16SR/0qGUNd5488I/0qGUNd5488I_0025.wav")
IPython.display.Audio("/home/amitli/Datasets/MyVoiceTests_16SR/-3dd58g7rQQ/-3dd58g7rQQ_0000.wav")

In [ ]:
#file_name   = "/home/amitli/Datasets/MyVoiceTests_16SR/-3dd58g7rQQ/-3dd58g7rQQ_0000.wav"
#file_name    = "/home/amitli/Datasets/MyVoiceTests_16SR/0qGUNd5488I/0qGUNd5488I_0011.wav"
file_name    = "/home/amitli/Datasets/MyVoiceTests_16SR/0qGUNd5488I/0qGUNd5488I_0025.wav"
simple_text = Get_Whisper_Text(whisper_model, whisper_processor, file_name, "cpu")

In [ ]:
simple_text

In [ ]:
result = whisper_model.transcribe(file_name, task='translate', beam_size=8, patience=2, fp16 = False)

In [ ]:
result['text']

In [ ]:
# audio = whisper.load_audio(file_name)
# audio = whisper.pad_or_trim(audio)
# mel   = whisper.log_mel_spectrogram(audio).to(whisper_model.device)

# # decode the audio
# options = whisper.DecodingOptions(language = 'he', beam_size=8, patience=2, fp16 = False)    
# result = whisper.decode(whisper_model, mel, options)


In [ ]:
from pyctcdecode       import build_ctcdecoder
from transformers      import Wav2Vec2ForCTC, Wav2Vec2Processor
from torchaudio.utils  import download_asset

import torch
import librosa

processor        = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model            = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")

FILE_NAME        = "tutorial-assets/Lab41-SRI-VOiCES-src-sp0307-ch127535-sg0042.wav"
SPEECH_FILE      = download_asset(FILE_NAME)

speech, sr       = librosa.load(SPEECH_FILE, sr=16000)
input_values     = processor(speech, sampling_rate=16000, return_tensors="pt").input_values

with torch.no_grad():
    logits           = model(input_values).logits

vocabulary       = list(processor.tokenizer.get_vocab().keys())
decoder          = build_ctcdecoder(vocabulary)
text             = decoder.decode(logits[0].numpy(), beam_width=5)
text

In [ ]:
from pyctcdecode       import build_ctcdecoder
from transformers      import Wav2Vec2ForCTC, Wav2Vec2Processor
from torchaudio.utils  import download_asset

import torch
import librosa

model_name       = "imvladikon/wav2vec2-xls-r-300m-hebrew"
processor        = Wav2Vec2Processor.from_pretrained(model_name)
model            = Wav2Vec2ForCTC.from_pretrained(model_name)

speech, sr       = librosa.load("/home/amitli/Datasets/MyVoiceTests_16SR/-3dd58g7rQQ/-3dd58g7rQQ_0000.wav", sr=16000)

inputs           = processor(speech, sampling_rate=16000, return_tensors="pt")
input_values     = inputs.input_values.to("cpu")     
with torch.no_grad():
    logits           = model(input_values).logits

vocabulary       = list(processor.tokenizer.get_vocab().keys())
decoder          = build_ctcdecoder(vocabulary)

hebrew_text      = decoder.decode(logits[0].numpy(), beam_width=1)
print(decode_hebrew(hebrew_text))

# if True:
#     predicted_ids    = torch.argmax(logits, dim=-1)
#     transcription    = processor.decode(predicted_ids[0])
#     hebrew_text      = decode_hebrew(transcription)
#     print(hebrew_text)

In [ ]:
logits.shape

In [ ]:
decoder.decode_beams(logits[0].numpy(),beam_width=1)

In [ ]:
predicted_ids    = torch.argmax(logits, dim=-1)
transcription    = processor.decode(predicted_ids[0])
hebrew_text      = decode_hebrew(transcription)
print(hebrew_text)

In [ ]:
vocabulary